In [17]:
import pandas as pd

df = pd.read_csv("../data/cleaned/cleaned_data.csv")
df['Order Date'] = pd.to_datetime(df['Order Date'])

# RFM Analysis

In [18]:
snapshot_date = df['Order Date'].max() + pd.Timedelta(days=1)

In [19]:
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days,
    'Order ID': 'nunique',
    'Sales': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
AA-10315,185,5,5563.560
AA-10375,20,9,1056.390
AA-10480,260,4,1790.512
AA-10645,56,6,5086.935
AB-10015,416,3,886.156


In [20]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

In [21]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [22]:
def segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return "Champions"
    elif row['R_score'] >= 3:
        return "Loyal Customers"
    elif row['R_score'] <= 2:
        return "At Risk"
    else:
        return "Others"

rfm['Segment'] = rfm.apply(segment, axis=1)

In [23]:
rfm['Segment'].value_counts()

Segment
At Risk            316
Loyal Customers    308
Champions          169
Name: count, dtype: int64

# Regional Analysis

In [24]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
region_sales

Region
West       710219.6845
East       669518.7260
Central    492646.9132
South      389151.4590
Name: Sales, dtype: float64

In [25]:
region_orders = df.groupby('Region')['Order ID'].nunique()

In [26]:
region_aov = region_sales / region_orders
region_aov

Region
Central    426.165150
East       489.056776
South      480.433900
West       447.523431
dtype: float64

In [27]:
region_category = df.groupby(['Region','Category'])['Sales'].sum()
region_category

Region   Category       
Central  Furniture          160317.4622
         Office Supplies    163590.2430
         Technology         168739.2080
East     Furniture          206461.3880
         Office Supplies    199940.8110
         Technology         263116.5270
South    Furniture          116531.4800
         Office Supplies    124424.7710
         Technology         148195.2080
West     Furniture          245348.2455
         Office Supplies    217466.5090
         Technology         247404.9300
Name: Sales, dtype: float64

In [28]:
top_cities = df.groupby('City')['Sales'].sum().sort_values(ascending=False).head(10)

In [29]:
region_segment = df.groupby(['Region','Segment'])['Sales'].sum()
region_segment

Region   Segment    
Central  Consumer       250210.5220
         Corporate      152031.4968
         Home Office     90404.8944
East     Consumer       347906.6050
         Corporate      195897.4250
         Home Office    125714.6960
South    Consumer       194702.2130
         Corporate      120546.8745
         Home Office     73902.3715
West     Consumer       355241.1910
         Corporate      220018.2785
         Home Office    134960.2150
Name: Sales, dtype: float64

**Insight:** “Corporate segment dominates in the West region”